<a href="https://colab.research.google.com/github/HiveCase/Deep-Learning/blob/main/Bonus%20Assignments/Bonus_Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Data Loading & Preprocessing

In [16]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [17]:
torch.manual_seed(42)

In [19]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),
                         (0.5,0.5,0.5))
])

In [22]:
train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
    )
test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

In [23]:
train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=100,
    shuffle=True
)

In [24]:
test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=100,
    shuffle=False
)

##Question 1: How many images are there in the training dataset?

In [27]:
print(f"{len(train_dataset)} images are there in the training dataset")

50000 images are there in the training dataset


##Question 2: What is the size of each image in the dataset (width, height, number of filters)?

torch.Size([2, 32, 32])

torch.Size([2, 64, 64])

torch.Size([3, 32, 32])

torch.Size([3, 64, 64])

In [28]:
image, label = train_dataset[0]
print(f"Size of each image in the dataset is {image.shape}")

Size of each image in the dataset is torch.Size([3, 32, 32])


#Model Building

In [30]:
import torch.nn as nn
import torch.nn.functional as F

In [31]:
class Net(nn.Module):
  def __init__(self):
    super(Net,self).__init__()
    # Convolution layers
    self.conv1 = nn.Conv2d(3,6,5)
    self.pool = nn.MaxPool2d(2,2)
    self.conv2 = nn.Conv2d(6,16,5)
    # fully connected layer
    self.fc1 = nn.Linear(400,120)
    self.fc2 = nn.Linear(120,84)
    self.fc3 = nn.Linear(84,10)
  def forward(self,x):
    x = self.pool(F.relu(self.conv1(x)))
    x = self.pool(F.relu(self.conv2(x)))
    x = x.view(-1,400)
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.fc3(x)
    return x

##Question 3: Enter the total number of parameters in the Net class, considering both weights and biases?

In [32]:
model = Net()
total_params = sum(p.numel() for p in model.parameters())
print(f"The total number of parameters in the Net class, considering both weights and biases are: {total_params}")

The total number of parameters in the Net class, considering both weights and biases are: 62006


##Question 4:Initilaize the loss function as Cross entropy loss. Initialize the Optimizer as stochastic gradient descent (SGD) with a momentum of  0.9 and a learning rate of 0.01. Run a forward pass through the model using the training data. What is the initial loss value?


In [33]:
import torch.optim as optim

In [34]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(),lr=0.01,momentum=0.9)

In [37]:
input, labels = next(iter(train_dataloader))
outputs = model(input)
loss = criterion(outputs,labels)
print(f"Initial loss value: {round(loss.item(),3)}")

Initial loss value: 2.306


#Model Training


Let us now train the model.

Loop over the dataset for two epochs:

Use a for loop with range(2) to loop over the dataset for 2 epochs.

Use trainloader to iterate over the training dataset.

For each batch of data:

	Zero Gradients: We start each iteration by setting the gradients to zero. Gradients accumulate by default, and we need to clear them before each backward pass.
	Making Predictions: The model makes predictions based on the input data.
	Computing Loss: We calculate the Cross entropy loss using the predictions.
	Backpropagation: Calling loss.backward() computes the gradient of the loss function with respect to each parameter in the model.
	Updating Parameters: The optimizer updates the model's parameters based on the gradients. This step moves the model's parameters closer to the values that minimize the loss function.

##Question 5: What is the loss value after 2 epochs? One epoch is training the network using entire training dataset.


In [39]:
for epoch in range(2):
  running_loss = 0.0
  for inputs, labels in train_dataloader:
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs,labels)
    loss.backward()
    optimizer.step()
    running_loss +=loss
  print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_dataloader)}")

Epoch 1, Loss: 1.700535774230957
Epoch 2, Loss: 1.405579924583435


#Model Evaluation

Your task is to evaluate the performance of the trained neural network on the test dataset. Follow the steps below:

	Use torch.no_grad() to disable gradient tracking during testing.
	Iterate over the testloader to get batches of test data.

For each batch of data:

	Extract the images and their corresponding labels.
	Perform a forward pass through the network to get the predicted outputs.
	Get the predicted labels by taking the maximum value along the predicted output tensor.
	Update the total count of test samples.
	Count the number of correct predictions and update the correct variable. Calculate the accuracy of the network on the test dataset. Provide the answer in range [0,1]


In [42]:
correct = 0
total = 0
with torch.no_grad():
  for images, labels in test_dataloader:
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)
    total+=labels.size(0)
    correct += (predicted==labels).sum().item()
accuracy = correct/total

In [43]:
print(f"Accuracy of the network: {accuracy}")

Accuracy of the network: 0.527
